# LeetCode #106: Construct Binary Tree from Inorder and Postorder Traversal

https://leetcode.com/problems/construct-binary-tree-from-inorder-and-postorder-traversal/

## Comparison of Approaches

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Recursive + Linear Search | O(n²) | O(n) | Scan inorder to find root each call |
| **Recursion + HashMap ★** | **O(n)** | **O(n)** | O(1) lookup; decrement postIndex right-first |

---

## Understanding the Methods

### Brute Force
At each call, scan the inorder array for the root (postorder[-1]). O(n) per call × O(n) calls = O(n²).

### Recursion + HashMap (Optimal ★)
Key: `postorder[postIndex]` is the current subtree's root. Decrement `postIndex` after use. Look up root in inorder map O(1) to split. **Critical:** build right subtree before left (postorder is left-right-root, so reading right-to-left gives root, then right subtree, then left).

## Constraints
- 1 ≤ n ≤ 3000
- No duplicate values

## Solutions

### C#

In [ ]:
public class Solution {
    private Dictionary<int,int> inorderMap;
    private int[] postorder;
    private int postIndex;

    public TreeNode BuildTree(int[] inorder, int[] postorder) {
        this.postorder  = postorder;
        this.postIndex  = postorder.Length - 1;
        this.inorderMap = new Dictionary<int,int>();
        for (int i = 0; i < inorder.Length; i++)
            inorderMap[inorder[i]] = i;
        return Build(0, inorder.Length - 1);
    }

    private TreeNode Build(int inStart, int inEnd) {
        if (inStart > inEnd) return null;
        int rootVal = postorder[postIndex--];
        int inRoot  = inorderMap[rootVal];
        var node    = new TreeNode(rootVal);
        node.right  = Build(inRoot + 1, inEnd);    // right BEFORE left
        node.left   = Build(inStart,    inRoot - 1);
        return node;
    }
}

### Python

In [ ]:
class Solution:
    def buildTree(self, inorder: list[int], postorder: list[int]):
        in_map     = {v: i for i, v in enumerate(inorder)}
        post_index = [len(postorder) - 1]  # mutable ref

        def build(in_start, in_end):
            if in_start > in_end:
                return None
            root_val        = postorder[post_index[0]]
            post_index[0]  -= 1
            in_root         = in_map[root_val]
            node            = TreeNode(root_val)
            node.right      = build(in_root + 1, in_end)    # right first
            node.left       = build(in_start,    in_root - 1)
            return node

        return build(0, len(inorder) - 1)

### Go

In [ ]:
func buildTree(inorder []int, postorder []int) *TreeNode {
    inMap      := make(map[int]int)
    for i, v  := range inorder { inMap[v] = i }
    postIndex  := len(postorder) - 1

    var build func(is, ie int) *TreeNode
    build = func(is, ie int) *TreeNode {
        if is > ie { return nil }
        rootVal := postorder[postIndex]; postIndex--
        inRoot  := inMap[rootVal]
        node    := &TreeNode{Val: rootVal}
        node.Right = build(inRoot+1, ie)
        node.Left  = build(is, inRoot-1)
        return node
    }
    return build(0, len(inorder)-1)
}

### Rust

In [ ]:
use std::rc::Rc;
use std::cell::RefCell;
use std::collections::HashMap;

impl Solution {
    pub fn build_tree(inorder: Vec<i32>, postorder: Vec<i32>) -> Option<Rc<RefCell<TreeNode>>> {
        let in_map: HashMap<i32,usize> = inorder.iter().enumerate().map(|(i,&v)|(v,i)).collect();
        let mut pi = postorder.len() as i32 - 1;

        fn build(
            postorder: &[i32], pi: &mut i32,
            in_map: &HashMap<i32,usize>, is: i32, ie: i32,
        ) -> Option<Rc<RefCell<TreeNode>>> {
            if is > ie || *pi < 0 { return None; }
            let root_val = postorder[*pi as usize]; *pi -= 1;
            let in_root  = in_map[&root_val] as i32;
            let node = Rc::new(RefCell::new(TreeNode::new(root_val)));
            node.borrow_mut().right = build(postorder, pi, in_map, in_root+1, ie);
            node.borrow_mut().left  = build(postorder, pi, in_map, is, in_root-1);
            Some(node)
        }
        let n = inorder.len() as i32;
        build(&postorder, &mut pi, &in_map, 0, n-1)
    }
}

## Example Scenarios

### 1. Common Case — Standard Tree
**Input:** `inorder=[9,3,15,20,7], postorder=[9,15,7,20,3]`
Root=3 (postorder[-1]). inRoot=1. Right: inorder[2..4] → root=20. Left: inorder[0..0] → root=9.
**Output:** `[3,9,20,null,null,15,7]`

### 2. Slightly Complex — Right-Skewed
**Input:** `inorder=[1,2,3], postorder=[1,2,3]`
Root=3. inRoot=2, rightSize=0. Left subtree: inorder[0..1], postorder scanning backwards.
**Output:** `[3,2,null,1]` (left-skewed in reverse)

### 3. Edge Case: Time Factor — Single Node
**Input:** `inorder=[1], postorder=[1]`
postIndex=0, root=1, inRoot=0, both children return null.
**Output:** `[1]`

### 4. Edge Case: Space Factor — Left-Skewed 3000 Nodes
**Input:** Fully left-skewed tree.
Postorder has root last. postIndex decrements from n-1 to 0. Stack depth O(n).
**Output:** Correct left-skewed tree

### 5. Almost-Impossible but Plausible — Two Nodes, Right Child Only
**Input:** `inorder=[1,2], postorder=[2,1]`
Root=1 (postorder[1]). inRoot=0. rightSize=1 → right=2. Left: inStart>inEnd → null.
**Output:** `[1,null,2]`

![image.png](attachment:image.png)